# SAHA-AL Benchmark — Final Evaluation Run

**What this notebook produces:**
- Task 2: Anonymization quality for 9 systems (8 existing + 1 new LLM baseline)
- Task 1: Detection P/R/F1 for rule-based systems
- Task 3: Privacy attacks (CRR-3, ERA, UAC, LRR) for 3 representative systems
- Failure taxonomy for 4 representative systems
- Pareto frontier across all 9 systems
- 4 publication-quality figures

**Estimated time:** ~90 min total on T4 GPU (mostly LLM baseline generation)

---
## 1. Setup

In [ ]:
!pip install -q sentence-transformers faker bert_score spacy presidio-analyzer presidio-anonymizer
!pip install -q bitsandbytes accelerate
!python -m spacy download en_core_web_lg -q

In [ ]:
import os

if not os.path.exists('repo'):
    !git clone https://github.com/hanara2112/pii_identification_and_replacement.git repo
else:
    !cd repo && git pull origin main

os.chdir('repo/benchmark')
os.makedirs('results', exist_ok=True)
os.makedirs('Results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print('Data files:'); !ls data/
print('\nPrediction files:'); !ls predictions/

---
## 2. Task 2: Re-evaluate All 8 Existing Systems

Ensures consistent entity types across all evaluations.

In [ ]:
seq2seq = ['bart-base-pii', 't5-small-pii', 'flan-t5-small-pii', 'distilbart-pii', 't5-efficient-tiny-pii']
rule_based = ['regex', 'spacy', 'presidio']

for model in seq2seq:
    print(f'\n{"="*60}\n  {model}\n{"="*60}')
    !python -m eval.eval_anonymization \
        --gold data/test.jsonl \
        --pred predictions/predictions_{model}.jsonl \
        --output Results/eval_anon_{model}.json \
        --print-types --skip-nli

for mode in rule_based:
    print(f'\n{"="*60}\n  {mode}\n{"="*60}')
    !python -m eval.eval_anonymization \
        --gold data/test.jsonl \
        --pred predictions/{mode}_predictions.jsonl \
        --output Results/eval_anon_{mode}.json \
        --print-types --skip-nli

---
## 3. Task 1: PII Detection

Span-level precision/recall/F1 (exact, partial, type-aware) for the 3 rule-based detectors.

In [ ]:
for mode in ['regex', 'spacy', 'presidio']:
    print(f'\n{"="*60}\n  Detection: {mode}\n{"="*60}')
    !python -m eval.eval_detection \
        --gold data/test.jsonl \
        --pred predictions/{mode}_spans.jsonl \
        --output results/eval_det_{mode}.json

---
## 4. NEW: LLM Zero-Shot Baseline (Qwen2.5-7B-Instruct)

Fills the missing "LLM baseline" gap. Runs locally with 4-bit quantization (~4GB VRAM).

**~45-60 min on T4 for 3600 records.** Start this early, other cells can wait.

In [ ]:
!python -m baselines.llm_baseline \
    --gold data/test.jsonl \
    --output predictions/predictions_qwen7b.jsonl \
    --local \
    --local-model 'Qwen/Qwen2.5-7B-Instruct'

In [ ]:
# Evaluate the LLM baseline
!python -m eval.eval_anonymization \
    --gold data/test.jsonl \
    --pred predictions/predictions_qwen7b.jsonl \
    --output Results/eval_anon_qwen7b.json \
    --print-types --skip-nli

---
## 5. Task 3: Privacy Risk (3 representative systems)

Runs CRR-3 + ERA + UAC on:
- **BART** (best seq2seq)
- **Presidio** (production tool, weakest privacy)
- **Qwen-7B** (LLM baseline, new)

ERA uses sentence-transformers (GPU). ~5 min per system.

In [ ]:
privacy_systems = {
    'bart':     'predictions/predictions_bart-base-pii.jsonl',
    'presidio': 'predictions/presidio_predictions.jsonl',
    'qwen7b':   'predictions/predictions_qwen7b.jsonl',
}

for name, pred_path in privacy_systems.items():
    if not os.path.exists(pred_path):
        print(f'  [SKIP] {pred_path} not found')
        continue
    print(f'\n{"="*60}\n  Privacy: {name}\n{"="*60}')
    !python -m eval.eval_privacy \
        --gold data/test.jsonl \
        --pred {pred_path} \
        --train data/train.jsonl \
        --output results/eval_privacy_{name}.json \
        --era-sample 500 \
        --skip-lrr

---
## 6. LRR: LLM Re-identification Attack

Uses Qwen2.5-7B-Instruct (local, 4-bit) as the adversary.

Runs on **BART** and **Presidio** — the contrast is the key finding:
- BART (seq2seq replacement) should resist LRR
- Presidio (pattern masking) should be more vulnerable

In [ ]:
# LRR on BART (expect ~0% — seq2seq is resistant)
!python -m eval.eval_privacy \
    --gold data/test.jsonl \
    --pred predictions/predictions_bart-base-pii.jsonl \
    --train data/train.jsonl \
    --output results/eval_privacy_bart_lrr.json \
    --skip-era \
    --lrr-local \
    --lrr-model 'Qwen/Qwen2.5-7B-Instruct' \
    --lrr-sample 200

In [ ]:
# LRR on Presidio (expect HIGHER — masking is more vulnerable)
!python -m eval.eval_privacy \
    --gold data/test.jsonl \
    --pred predictions/presidio_predictions.jsonl \
    --train data/train.jsonl \
    --output results/eval_privacy_presidio_lrr.json \
    --skip-era \
    --lrr-local \
    --lrr-model 'Qwen/Qwen2.5-7B-Instruct' \
    --lrr-sample 200

### 6b. Chain-of-Thought LRR (BART only — robustness test)

Stronger prompt with explicit reasoning instructions.
If BART still resists → claim is robust.

In [ ]:
import json, torch
from difflib import SequenceMatcher
from eval.utils import load_jsonl
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

COT_PROMPT = """You are a privacy researcher testing anonymization quality. Your goal is to REVERSE the anonymization and recover the ORIGINAL personally identifiable information.

Think step by step:
1. Identify which parts look like fake replacements (names, numbers, addresses, emails)
2. Look for contextual clues (job titles, locations, relationships) that constrain the original
3. Use world knowledge to narrow down the most likely originals

Anonymized text: {text}

For each replaced entity, output your best guess: REPLACEMENT -> YOUR_GUESS (one per line).
Think carefully before answering."""

model_name = 'Qwen/Qwen2.5-7B-Instruct'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map='auto')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gold = load_jsonl('data/test.jsonl')[:200]
preds = load_jsonl('predictions/predictions_bart-base-pii.jsonl')[:200]

exact, fuzzy, total = 0, 0, 0
for i, (g, p) in enumerate(zip(gold, preds)):
    ents = {e['text'] for e in g.get('entities', []) if e.get('text')}
    if not ents:
        continue
    prompt = COT_PROMPT.format(text=p['anonymized_text'])
    messages = [{'role': 'user', 'content': prompt}]
    full_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(full_prompt, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True,
                             top_p=0.9, pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    guesses = {l.split('->')[-1].strip() for l in raw.split('\n') if '->' in l}

    for e in ents:
        total += 1
        if e in guesses:
            exact += 1
        elif any(SequenceMatcher(None, e.lower(), gg.lower()).ratio() > 0.8 for gg in guesses):
            fuzzy += 1
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/200: exact={exact}, fuzzy={fuzzy}, total={total}')

cot_results = {
    'lrr_cot_exact': round(exact / total * 100, 2) if total else 0,
    'lrr_cot_fuzzy': round((exact + fuzzy) / total * 100, 2) if total else 0,
    'total': total,
    'model': model_name,
    'prompt': 'chain-of-thought'
}
with open('results/eval_lrr_cot_bart.json', 'w') as f:
    json.dump(cot_results, f, indent=2)

print(f'\nCoT LRR (BART): exact={cot_results["lrr_cot_exact"]:.2f}%, '
      f'fuzzy={cot_results["lrr_cot_fuzzy"]:.2f}% (n={total})')

---
## 7. Failure Taxonomy (4 representative systems)

Only running on BART (best seq2seq), spaCy (mid), Presidio (production), Regex (worst).

Flan-T5/T5-small/DistilBART are nearly identical to BART — no insight gained.

In [ ]:
taxonomy_systems = {
    'bart-base-pii': 'predictions/predictions_bart-base-pii.jsonl',
    'spacy':         'predictions/spacy_predictions.jsonl',
    'presidio':      'predictions/presidio_predictions.jsonl',
    'regex':         'predictions/regex_predictions.jsonl',
}

for name, pred_path in taxonomy_systems.items():
    print(f'\n{"="*60}\n  Failure Taxonomy: {name}\n{"="*60}')
    !python -m analysis.failure_taxonomy \
        --gold data/test.jsonl \
        --pred {pred_path} \
        --output results/failure_{name}.json

---
## 8. Pareto Frontier (all systems)

Privacy (1-ELR) vs Utility (BERTScore/100) — now including LLM baseline.

In [ ]:
import json, os

eval_files = {
    'regex':         'Results/eval_anon_regex.json',
    'spacy':         'Results/eval_anon_spacy.json',
    'presidio':      'Results/eval_anon_presidio.json',
    'bart-base':     'Results/eval_anon_bart-base-pii.json',
    'flan-t5-small': 'Results/eval_anon_flan-t5-small-pii.json',
    'distilbart':    'Results/eval_anon_distilbart-pii.json',
    't5-small':      'Results/eval_anon_t5-small-pii.json',
    't5-eff-tiny':   'Results/eval_anon_t5-efficient-tiny-pii.json',
}
if os.path.exists('Results/eval_anon_qwen7b.json'):
    eval_files['qwen-7b'] = 'Results/eval_anon_qwen7b.json'

all_eval = {}
for name, path in eval_files.items():
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        all_eval[name] = {'elr': d['elr'], 'bertscore': d['bertscore_f1']}

with open('results/all_eval_results.json', 'w') as f:
    json.dump(all_eval, f, indent=2)

print(f'Systems in Pareto analysis: {list(all_eval.keys())}')

!python -m analysis.pareto_frontier \
    --results results/all_eval_results.json \
    --output results/pareto_analysis.json \
    --plot figures/pareto_frontier.png

In [ ]:
from IPython.display import Image, display
if os.path.exists('figures/pareto_frontier.png'):
    display(Image('figures/pareto_frontier.png'))

---
## 9. Generate All Figures

Produces 4 publication-quality plots:
1. `task2_comparison.png` — ELR + BERTScore grouped bar chart
2. `attack_heatmap.png` — Systems × Attack types recovery heatmap
3. `failure_taxonomy.png` — Stacked bar of failure categories
4. `detection_recall.png` — Per-type recall heatmap

In [ ]:
!python -m analysis.plot_results \
    --results-dir results \
    --eval-dir Results \
    --output-dir figures

In [ ]:
from IPython.display import Image, display
import os
for fig in ['task2_comparison.png', 'attack_heatmap.png', 'failure_taxonomy.png',
            'detection_recall.png', 'pareto_frontier.png']:
    path = f'figures/{fig}'
    if os.path.exists(path):
        print(f'\n--- {fig} ---')
        display(Image(path))

---
## 10. Final Summary Tables

### What's useful vs what to cut

**KEEP (report in paper):**
- Task 2 main table (all 9 systems) — core benchmark result
- Task 1 detection (3 rule-based) — shows detection gap drives anonymization gap
- Attack-system interaction table (BART vs Presidio vs LLM) — THE central novelty
- LRR standard + CoT (BART only) — robustness evidence
- Pareto frontier — Privacy-Utility tradeoff visualization
- Failure taxonomy (4 systems) — qualitative error analysis

**CUT (pointless / overkill):**
- Flan-T5 privacy: CRR=34.94% vs BART 34.62% → identical, redundant
- Type Confusion: 0% across ALL systems → nothing to report
- Over-Masking in taxonomy: 0% across ALL systems (OMR metric is different, keep that)
- T5-small / DistilBART failure taxonomy: same pattern as BART, no new insight
- spaCy LRR: too many API failures, unreliable
- Per-type ELR for every model: only BART (best) and Presidio (worst) matter

In [ ]:
import json, os

print('=' * 85)
print('  SAHA-AL BENCHMARK — FINAL RESULTS')
print('=' * 85)

# ── Task 2 ──
eval_files = {
    'BART-base':     'Results/eval_anon_bart-base-pii.json',
    'Flan-T5-small': 'Results/eval_anon_flan-t5-small-pii.json',
    'T5-small':      'Results/eval_anon_t5-small-pii.json',
    'DistilBART':    'Results/eval_anon_distilbart-pii.json',
    'T5-eff-tiny':   'Results/eval_anon_t5-efficient-tiny-pii.json',
    'Qwen-7B (LLM)':'Results/eval_anon_qwen7b.json',
    'spaCy+Faker':   'Results/eval_anon_spacy.json',
    'Presidio':      'Results/eval_anon_presidio.json',
    'Regex+Faker':   'Results/eval_anon_regex.json',
}

print(f'\n  Task 2: Text Anonymization Quality')
print('-' * 85)
print(f'{"System":20s} {"ELR↓":>8s} {"TokRec↑":>8s} {"OMR↓":>8s} {"FPR↑":>8s} {"BERT↑":>8s}')
print('-' * 85)
for name, path in eval_files.items():
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        print(f'{name:20s} {d.get("elr",0):7.2f}% {d.get("token_recall",0):7.2f}% '
              f'{d.get("over_masking_rate",0):7.2f}% {d.get("format_preservation_rate",0):7.2f}% '
              f'{d.get("bertscore_f1",0):7.2f}')

# ── Task 1 ──
print(f'\n  Task 1: PII Detection (Span-Level)')
print('-' * 85)
print(f'{"System":20s} {"Exact F1":>10s} {"Partial F1":>12s} {"Type-Aware F1":>14s}')
print('-' * 85)
for mode in ['regex', 'spacy', 'presidio']:
    path = f'results/eval_det_{mode}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        print(f'{mode:20s} {d["exact"]["f1"]:9.2f}% {d["partial"]["f1"]:11.2f}% '
              f'{d["type_aware"]["f1"]:13.2f}%')

# ── Task 3: Attack Interaction Matrix (THE central table) ──
print(f'\n  Task 3: Privacy Under Attack — System × Attack Matrix')
print('-' * 85)
print(f'{"System":20s} {"ELR":>7s} {"CRR-3":>7s} {"ERA@1":>7s} {"ERA@5":>7s} {"UAC":>7s} {"LRR":>7s}')
print('-' * 85)

for label, priv_f, anon_f, lrr_f in [
    ('BART-base',  'eval_privacy_bart.json',     'eval_anon_bart-base-pii.json',  'eval_privacy_bart_lrr.json'),
    ('Presidio',   'eval_privacy_presidio.json',  'eval_anon_presidio.json',       'eval_privacy_presidio_lrr.json'),
    ('Qwen-7B',    'eval_privacy_qwen7b.json',    'eval_anon_qwen7b.json',         None),
]:
    priv_path = f'results/{priv_f}'
    anon_path = f'Results/{anon_f}'
    if not os.path.exists(priv_path):
        continue
    with open(priv_path) as f:
        priv = json.load(f)
    anon = json.load(open(anon_path)) if os.path.exists(anon_path) else {}
    era = priv.get('era') or {}

    lrr_val = 'N/A'
    if lrr_f:
        lrr_path = f'results/{lrr_f}'
        if os.path.exists(lrr_path):
            lrr_data = json.load(open(lrr_path))
            lrr_inner = lrr_data.get('lrr') or {}
            lrr_val = f'{lrr_inner.get("lrr_exact", 0):.2f}%'

    print(f'{label:20s} {anon.get("elr",0):6.2f}% {priv.get("crr3",0):6.2f}% '
          f'{era.get("era_top1",0):6.2f}% {era.get("era_top5",0):6.2f}% '
          f'{priv.get("uac",0):6.2f}% {lrr_val:>7s}')

# ── CoT LRR ──
cot_path = 'results/eval_lrr_cot_bart.json'
if os.path.exists(cot_path):
    with open(cot_path) as f:
        cot = json.load(f)
    print(f'\n  LRR Robustness (BART, Chain-of-Thought): '
          f'exact={cot["lrr_cot_exact"]:.2f}%, fuzzy={cot["lrr_cot_fuzzy"]:.2f}% (n={cot["total"]})')

# ── Failure Taxonomy ──
print(f'\n  Failure Taxonomy')
print('-' * 85)
cats = ['clean', 'full_leak', 'ghost_leak', 'boundary', 'format_break']
cat_labels = ['Clean', 'Full Leak', 'Ctx Retain', 'Boundary', 'Fmt Break']
print(f'{"System":20s}', end='')
for cl in cat_labels:
    print(f' {cl:>11s}', end='')
print()
print('-' * 85)
for name in ['bart-base-pii', 'spacy', 'presidio', 'regex']:
    path = f'results/failure_{name}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        counts = d['counts']
        total = sum(counts.values())
        print(f'{name:20s}', end='')
        for c in cats:
            v = counts.get(c, 0)
            print(f' {v/total*100:10.1f}%', end='')
        print()

# ── Pareto ──
pareto_path = 'results/pareto_analysis.json'
if os.path.exists(pareto_path):
    with open(pareto_path) as f:
        pareto = json.load(f)
    print(f'\n  Pareto-optimal systems: {pareto["pareto_optimal"]}')

# ── File listing ──
print(f'\n{"=" * 85}')
print('  Output Files')
print('=' * 85)
for d in ['Results', 'results', 'figures']:
    if os.path.exists(d):
        for fn in sorted(os.listdir(d)):
            fp = os.path.join(d, fn)
            print(f'  {fp:55s} ({os.path.getsize(fp):,} bytes)')

---
## 11. Download Results

In [ ]:
import shutil, os

# Merge Results/ into results/
for f in os.listdir('Results'):
    shutil.copy2(f'Results/{f}', f'results/{f}')

# Copy figures
os.makedirs('/content/saha_al_output', exist_ok=True)
shutil.copytree('results', '/content/saha_al_output/results', dirs_exist_ok=True)
shutil.copytree('figures', '/content/saha_al_output/figures', dirs_exist_ok=True)
shutil.make_archive('/content/saha_al_final', 'zip', '/content/saha_al_output')

print('Download: /content/saha_al_final.zip')
print(f'Contains {len(os.listdir("/content/saha_al_output/results"))} result files + '
      f'{len(os.listdir("/content/saha_al_output/figures"))} figures')

try:
    from google.colab import files
    files.download('/content/saha_al_final.zip')
except:
    pass